In [22]:
# %pip install SPARQLWrapper pandas


## Person

In [ ]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# 2️⃣ Create 10-year ranges (keep if you still want chunking)
year_ranges = [(year, min(year + 0, 2026)) for year in range(1910, 2026, 1)]

for start_year, end_year in year_ranges:
    print(f"Fetching US citizens (IMDb required) for birth years {start_year} to {end_year}...")

    # 3️⃣ SPARQL query for this decade
    # - US citizen (Q30) OR Canada citizen (Q16)
    # - IMDb required (P345)
    # - Birthdate OPTIONAL (do not require it)
    # - Gender OPTIONAL (do not require it)
    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person ?personLabel
      ?citizenshipLabel
      ?genderLabel
      ?birthDate
      ?imdb
    WHERE {{
      ?person wdt:P31 wd:Q5;        # human
              wdt:P27 ?citizenship; # citizenship
              wdt:P345 ?imdb.       # IMDb ID (required)

      VALUES ?citizenship {{ wd:Q30 }}  # US
      hint:Prior hint:rangeSafe true.

      OPTIONAL {{ ?person wdt:P569 ?birthDate. }}  # birthdate optional
      OPTIONAL {{ ?person wdt:P21  ?gender. }}     # gender optional

      # If birthDate exists, filter it into the decade; if missing, keep it.
      FILTER(!BOUND(?birthDate) ||
             ("{start_year}-01-01T00:00:00Z"^^xsd:dateTime <= ?birthDate &&
              ?birthDate <= "{end_year}-12-31T23:59:59Z"^^xsd:dateTime))

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?birthDate ?person
    LIMIT 200000
    """

    sparql.setQuery(query)

    # 4️⃣ Execute
    results = sparql.query().convert()

    # 5️⃣ Parse results
    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "person": r["person"]["value"],
            "name": r.get("personLabel", {}).get("value"),
            "citizenship": r.get("citizenshipLabel", {}).get("value"),
            "gender": r.get("genderLabel", {}).get("value"),
            "birthDate": r.get("birthDate", {}).get("value"),  # may be missing
            "IMDb": r.get("imdb", {}).get("value"),
        })

    # 6️⃣ DataFrame
    df_people = pd.DataFrame(rows)

    # 7️⃣ Save CSV for this decade
    filename = f"wikidata_us_people_personal_info_{start_year}_{end_year}.csv"
    df_people.to_csv(filename, index=False)

    print(f"Saved {filename}, rows: {len(df_people)}\n")


Fetching US + CA citizens (IMDb required) for birth years 1910 to 1911...


HTTPError: HTTP Error 429: Too Many Requests

## CA imdb

In [ ]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

# -----------------------------
# Config: chunk by IMDb nm range
# -----------------------------
NM_MIN = 1          # e.g., nm0000001
NM_MAX = 9999999    # adjust to what you want as "max"
CHUNK_SIZE = 500000 # numeric range size per query (tune smaller if you hit timeouts)
LIMIT_PER_CHUNK = 200000  # SPARQL LIMIT (keep reasonable)
SLEEP_SEC = 1       # be nice to Wikidata

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

def fetch_chunk(nm_start: int, nm_end: int) -> pd.DataFrame:
    print(f"Fetching CA citizens with IMDb in nm[{nm_start}..{nm_end}] ...")

    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person ?personLabel
      ?citizenshipLabel
      ?genderLabel
      ?birthDate
      ?imdb
      ?nmNum
    WHERE {{
      ?person wdt:P31 wd:Q5;        # human
              wdt:P27 wd:Q16;       # Canada citizenship
              wdt:P345 ?imdb.       # IMDb ID required

      # Extract numeric part from "nm########"
      FILTER(STRSTARTS(STR(?imdb), "nm"))
      BIND(xsd:integer(SUBSTR(STR(?imdb), 3)) AS ?nmNum)

      FILTER(?nmNum >= {nm_start} && ?nmNum <= {nm_end})

      OPTIONAL {{ ?person wdt:P569 ?birthDate. }}  # birthdate optional
      OPTIONAL {{ ?person wdt:P21  ?gender. }}     # gender optional

      hint:Prior hint:rangeSafe true.

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?nmNum ?person
    LIMIT {LIMIT_PER_CHUNK}
    """

    sparql.setQuery(query)

    # Basic retry (Wikidata sometimes rate-limits)
    for attempt in range(1, 6):
        try:
            results = sparql.query().convert()
            break
        except Exception as e:
            if attempt == 5:
                raise
            wait = SLEEP_SEC * attempt
            print(f"Query failed (attempt {attempt}/5): {e}\nRetrying in {wait}s...")
            time.sleep(wait)

    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "person": r["person"]["value"],
            "name": r.get("personLabel", {}).get("value"),
            "citizenship": r.get("citizenshipLabel", {}).get("value"),
            "gender": r.get("genderLabel", {}).get("value"),
            "birthDate": r.get("birthDate", {}).get("value"),  # may be missing
            "IMDb": r.get("imdb", {}).get("value"),
            "nmNum": int(r.get("nmNum", {}).get("value")) if "nmNum" in r else None,
        })

    return pd.DataFrame(rows)

# -----------------------------
# Run: iterate nm ranges
# -----------------------------
for nm_start in range(NM_MIN, NM_MAX + 1, CHUNK_SIZE):
    nm_end = min(nm_start + CHUNK_SIZE - 1, NM_MAX)

    df_people = fetch_chunk(nm_start, nm_end)

    filename = f"unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_{nm_start}_{nm_end}.csv"
    df_people.to_csv(filename, index=False)
    print(f"Saved {filename}, rows: {len(df_people)}\n")

    time.sleep(SLEEP_SEC)


Fetching CA citizens with IMDb in nm[4450001..4950000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_4450001_4950000.csv, rows: 274

Fetching CA citizens with IMDb in nm[4950001..5450000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_4950001_5450000.csv, rows: 291

Fetching CA citizens with IMDb in nm[5450001..5950000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_5450001_5950000.csv, rows: 202

Fetching CA citizens with IMDb in nm[5950001..6450000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_5950001_6450000.csv, rows: 187

Fetching CA citizens with IMDb in nm[6450001..6950000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_6450001_6950000.csv, rows: 194

Fetching CA citizens with IMDb in nm[6950001..7450000] ...
Saved unprocessed/people_ca_imdb/wikidata_ca_people_imdb_nm_6950001_7450000.csv, rows: 189

Fetching CA citizens with IMDb in nm[7450001..7950000] ...
Saved unprocessed/people_ca_imdb/wi

## US imdb

In [11]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time

# -----------------------------
# Config: chunk by IMDb nm range
# -----------------------------
NM_MIN = 2340001          # e.g., 2340001
NM_MAX = 9999999
CHUNK_SIZE = 1000000      # reduce if timeout
LIMIT_PER_CHUNK = 200000 # reduce if timeout
SLEEP_SEC = 1

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

def fetch_chunk(nm_start: int, nm_end: int) -> pd.DataFrame:
    print(f"Fetching US citizens with IMDb in nm[{nm_start}..{nm_end}] ...")

    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person ?personLabel
      ?citizenshipLabel
      ?genderLabel
      ?birthDate
      ?imdb
      ?nmNum
    WHERE {{
      ?person wdt:P31 wd:Q5;        # human
              wdt:P27 wd:Q30;       # United States citizenship
              wdt:P345 ?imdb.       # IMDb ID required

      FILTER(STRSTARTS(STR(?imdb), "nm"))
      BIND(xsd:integer(SUBSTR(STR(?imdb), 3)) AS ?nmNum)

      FILTER(?nmNum >= {nm_start} && ?nmNum <= {nm_end})

      OPTIONAL {{ ?person wdt:P569 ?birthDate. }}
      OPTIONAL {{ ?person wdt:P21  ?gender. }}

      hint:Prior hint:rangeSafe true.

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?nmNum ?person
    LIMIT {LIMIT_PER_CHUNK}
    """

    sparql.setQuery(query)

    # Retry logic
    for attempt in range(1, 6):
        try:
            results = sparql.query().convert()
            break
        except Exception as e:
            if attempt == 5:
                raise
            wait = SLEEP_SEC * attempt
            print(f"Query failed (attempt {attempt}/5): {e}")
            print(f"Retrying in {wait}s...\n")
            time.sleep(wait)

    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "person": r["person"]["value"],
            "name": r.get("personLabel", {}).get("value"),
            "citizenship": r.get("citizenshipLabel", {}).get("value"),
            "gender": r.get("genderLabel", {}).get("value"),
            "birthDate": r.get("birthDate", {}).get("value"),
            "IMDb": r.get("imdb", {}).get("value"),
            "nmNum": int(r.get("nmNum", {}).get("value")) if "nmNum" in r else None,
        })

    return pd.DataFrame(rows)


# -----------------------------
# Run chunk loop
# -----------------------------
for nm_start in range(NM_MIN, NM_MAX + 1, CHUNK_SIZE):
    nm_end = min(nm_start + CHUNK_SIZE - 1, NM_MAX)

    df_people = fetch_chunk(nm_start, nm_end)

    filename = f"unprocessed/people_us_imdb/wikidata_ca_people_imdb_nm_{nm_start}_{nm_end}.csv"
    df_people.to_csv(filename, index=False)
    print(f"Saved {filename}, rows: {len(df_people)}\n")

    time.sleep(SLEEP_SEC)


Fetching US citizens with IMDb in nm[2340001..3340000] ...


OSError: Cannot save file into a non-existent directory: 'unprocessed/people_us_imdb'

## UK imdb

In [12]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time
import ssl
import urllib.request

# Disable SSL verification
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

opener = urllib.request.build_opener(urllib.request.HTTPSHandler(context=ssl_context))
urllib.request.install_opener(opener)

NM_MIN = 0
NM_MAX = 9999999
CHUNK_SIZE = 100000
LIMIT_PER_CHUNK = 200000
SLEEP_SEC = 1

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

def fetch_chunk(nm_start: int, nm_end: int) -> pd.DataFrame:
    print(f"Fetching UK citizens with IMDb in nm[{nm_start}..{nm_end}] ...")

    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person ?personLabel
      ?citizenshipLabel
      ?genderLabel
      ?birthDate
      ?imdb
      ?nmNum
    WHERE {{
      ?person wdt:P31 wd:Q5;
              wdt:P345 ?imdb.

      VALUES ?ukEntity {{ wd:Q145 wd:Q21 wd:Q22 wd:Q23 wd:Q26 }}
      ?person wdt:P27 ?ukEntity.

      FILTER(STRSTARTS(STR(?imdb), "nm"))
      BIND(xsd:integer(SUBSTR(STR(?imdb), 3)) AS ?nmNum)

      FILTER(?nmNum >= {nm_start} && ?nmNum <= {nm_end})

      OPTIONAL {{ ?person wdt:P569 ?birthDate. }}
      OPTIONAL {{ ?person wdt:P21  ?gender. }}

      hint:Prior hint:rangeSafe true.

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?nmNum ?person
    LIMIT {LIMIT_PER_CHUNK}
    """

    sparql.setQuery(query)

    for attempt in range(1, 6):
        try:
            results = sparql.query().convert()
            break
        except Exception as e:
            if attempt == 5:
                raise
            wait = SLEEP_SEC * attempt
            print(f"Query failed (attempt {attempt}/5): {e}")
            print(f"Retrying in {wait}s...\n")
            time.sleep(wait)

    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "person": r["person"]["value"],
            "name": r.get("personLabel", {}).get("value"),
            "citizenship": r.get("citizenshipLabel", {}).get("value"),
            "gender": r.get("genderLabel", {}).get("value"),
            "birthDate": r.get("birthDate", {}).get("value"),
            "IMDb": r.get("imdb", {}).get("value"),
            "nmNum": int(r.get("nmNum", {}).get("value")) if "nmNum" in r else None,
        })

    return pd.DataFrame(rows)


for nm_start in range(NM_MIN, NM_MAX + 1, CHUNK_SIZE):
    nm_end = min(nm_start + CHUNK_SIZE - 1, NM_MAX)

    df_people = fetch_chunk(nm_start, nm_end)

    filename = f"unprocessed/people_uk_imdb/wikidata_uk_people_imdb_nm_{nm_start}_{nm_end}.csv"
    df_people.to_csv(filename, index=False)
    print(f"Saved {filename}, rows: {len(df_people)}\n")

    time.sleep(SLEEP_SEC)

Fetching UK citizens with IMDb in nm[0..99999] ...
Saved unprocessed/people_uk_imdb/wikidata_uk_people_imdb_nm_0_99999.csv, rows: 2075

Fetching UK citizens with IMDb in nm[100000..199999] ...
Query failed (attempt 1/5): HTTP Error 504: Gateway Timeout
Retrying in 1s...



KeyboardInterrupt: 

## Relationship - Parent

In [14]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# 2️⃣ Create 10-year ranges from 1900 to 2026
year_ranges = [(year, min(year + 9, 2026)) for year in range(1970, 2026, 5)]

for start_year, end_year in year_ranges:
    print(f"Fetching data for birth years {start_year} to {end_year}...")

    # 3️⃣ SPARQL query for this decade
    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person
      ?father
      ?mother
    WHERE {{
      ?person wdt:P31 wd:Q5;        # human
              wdt:P27 wd:Q30;       # US citizen
              wdt:P569 ?birthDate.  # date of birth
      hint:Prior hint:rangeSafe true.

      FILTER(?birthDate >= "{start_year}-01-01T00:00:00Z"^^xsd:dateTime)
      FILTER(?birthDate <= "{end_year}-12-31T23:59:59Z"^^xsd:dateTime)

      OPTIONAL {{ ?person wdt:P22 ?father. }}  # father
      OPTIONAL {{ ?person wdt:P25 ?mother. }}  # mother

      # ✅ MUST have at least one parent
      FILTER(BOUND(?father) || BOUND(?mother))
    }}
    ORDER BY ?birthDate ?person
    LIMIT 200000
    """

    sparql.setQuery(query)

    # 4️⃣ Execute
    results = sparql.query().convert()

    # 5️⃣ Parse results
    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "personId": r["person"]["value"],
            "fatherId": r.get("father", {}).get("value"),
            "motherId": r.get("mother", {}).get("value"),
        })

    # 6️⃣ Convert to DataFrame
    df_parents = pd.DataFrame(rows)

    # 7️⃣ Save CSV for this decade
    filename = f"wikidata_us_people_parents_{start_year}_{end_year}.csv"
    df_parents.to_csv(filename, index=False)

    print(f"Saved {filename}, rows: {len(df_parents)}\n")


Fetching data for birth years 1970 to 1979...
Saved wikidata_us_people_parents_1970_1979.csv, rows: 1405

Fetching data for birth years 1975 to 1984...
Saved wikidata_us_people_parents_1975_1984.csv, rows: 1393

Fetching data for birth years 1980 to 1989...
Saved wikidata_us_people_parents_1980_1989.csv, rows: 1460

Fetching data for birth years 1985 to 1994...
Saved wikidata_us_people_parents_1985_1994.csv, rows: 1396

Fetching data for birth years 1990 to 1999...
Saved wikidata_us_people_parents_1990_1999.csv, rows: 1337

Fetching data for birth years 1995 to 2004...
Saved wikidata_us_people_parents_1995_2004.csv, rows: 1247

Fetching data for birth years 2000 to 2009...
Saved wikidata_us_people_parents_2000_2009.csv, rows: 865

Fetching data for birth years 2005 to 2014...
Saved wikidata_us_people_parents_2005_2014.csv, rows: 422

Fetching data for birth years 2010 to 2019...
Saved wikidata_us_people_parents_2010_2019.csv, rows: 249

Fetching data for birth years 2015 to 2024...
Sav

### Birthyear NULL Version - US 

In [83]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

print("Fetching US citizens with NULL birthyear...")

query = """
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX hint: <http://www.bigdata.com/queryHints#>

SELECT DISTINCT
  ?person
  ?father
  ?mother
WHERE {
  ?person wdt:P31 wd:Q5;        # human
          wdt:P27 wd:Q30.       # US citizen

  # 🚨 Birthdate does NOT exist
  FILTER NOT EXISTS { ?person wdt:P569 ?birthDate }

  OPTIONAL { ?person wdt:P22 ?father. }
  OPTIONAL { ?person wdt:P25 ?mother. }

  # Must have at least one parent
  FILTER(BOUND(?father) || BOUND(?mother))

  hint:Prior hint:rangeSafe true.
}
LIMIT 200000
"""

sparql.setQuery(query)

# Execute
results = sparql.query().convert()

# Parse
rows = []
for r in results["results"]["bindings"]:
    rows.append({
        "personId": r["person"]["value"],
        "fatherId": r.get("father", {}).get("value"),
        "motherId": r.get("mother", {}).get("value"),
    })

df_parents = pd.DataFrame(rows)

filename = "wikidata_us_people_null_birthyear_parents.csv"
df_parents.to_csv(filename, index=False)

print(f"Saved {filename}, rows: {len(df_parents)} ✅")


Fetching US citizens with NULL birthyear...
Saved wikidata_us_people_null_birthyear_parents.csv, rows: 1217 ✅


In [5]:
# If you run this in a notebook and hit ModuleNotFoundError:
# %pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# 2️⃣ Create 10-year ranges from 1900 to 2026
year_ranges = [(year, min(year + 9, 2026)) for year in range(1900, 2026, 10)]

for start_year, end_year in year_ranges:
    print(f"Fetching Canada citizens born {start_year} to {end_year}...")

    # 3️⃣ SPARQL query for this decade (Canada = wd:Q16)
    query = f"""
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      ?person
      ?father
      ?mother
    WHERE {{
      ?person wdt:P31 wd:Q5;        # human
              wdt:P27 wd:Q16;       # 🇨🇦 Canada citizen
              wdt:P569 ?birthDate.  # date of birth
      hint:Prior hint:rangeSafe true.

      FILTER(?birthDate >= "{start_year}-01-01T00:00:00Z"^^xsd:dateTime)
      FILTER(?birthDate <= "{end_year}-12-31T23:59:59Z"^^xsd:dateTime)

      OPTIONAL {{ ?person wdt:P22 ?father. }}  # father
      OPTIONAL {{ ?person wdt:P25 ?mother. }}  # mother

      # ✅ MUST have at least one parent
      FILTER(BOUND(?father) || BOUND(?mother))
    }}
    ORDER BY ?birthDate ?person
    LIMIT 200000
    """

    sparql.setQuery(query)

    # 4️⃣ Execute
    results = sparql.query().convert()

    # 5️⃣ Parse results
    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "personId": r["person"]["value"],
            "fatherId": r.get("father", {}).get("value"),
            "motherId": r.get("mother", {}).get("value"),
        })

    # 6️⃣ Convert to DataFrame
    df_parents = pd.DataFrame(rows)

    # 7️⃣ Save CSV for this decade
    filename = f"wikidata_ca_people_parents_{start_year}_{end_year}.csv"
    df_parents.to_csv(filename, index=False)

    print(f"Saved {filename}, rows: {len(df_parents)}\n")


Fetching Canada citizens born 1900 to 1909...
Saved wikidata_ca_people_parents_1900_1909.csv, rows: 150

Fetching Canada citizens born 1910 to 1919...
Saved wikidata_ca_people_parents_1910_1919.csv, rows: 125

Fetching Canada citizens born 1920 to 1929...
Saved wikidata_ca_people_parents_1920_1929.csv, rows: 166

Fetching Canada citizens born 1930 to 1939...
Saved wikidata_ca_people_parents_1930_1939.csv, rows: 150

Fetching Canada citizens born 1940 to 1949...
Saved wikidata_ca_people_parents_1940_1949.csv, rows: 205

Fetching Canada citizens born 1950 to 1959...
Saved wikidata_ca_people_parents_1950_1959.csv, rows: 223

Fetching Canada citizens born 1960 to 1969...
Saved wikidata_ca_people_parents_1960_1969.csv, rows: 230

Fetching Canada citizens born 1970 to 1979...
Saved wikidata_ca_people_parents_1970_1979.csv, rows: 184

Fetching Canada citizens born 1980 to 1989...
Saved wikidata_ca_people_parents_1980_1989.csv, rows: 177

Fetching Canada citizens born 1990 to 1999...
Saved wik

### Birthyear NULL Version - CA

In [84]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

print("Fetching CA citizens with NULL birthyear...")

query = """
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX hint: <http://www.bigdata.com/queryHints#>

SELECT DISTINCT
  ?person
  ?personLabel
  ?father
  ?mother
WHERE {
  ?person wdt:P31 wd:Q5;        # human
          wdt:P27 wd:Q16.       # Canada citizen

  # 🚨 Birthdate does NOT exist
  FILTER NOT EXISTS { ?person wdt:P569 ?birthDate }

  OPTIONAL { ?person wdt:P22 ?father. }
  OPTIONAL { ?person wdt:P25 ?mother. }

  # Must have at least one parent
  FILTER(BOUND(?father) || BOUND(?mother))

  hint:Prior hint:rangeSafe true.

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 200000
"""

sparql.setQuery(query)

# Execute
results = sparql.query().convert()

# Parse
rows = []
for r in results["results"]["bindings"]:
    rows.append({
        "personId": r["person"]["value"],
        "name": r.get("personLabel", {}).get("value"),
        "fatherId": r.get("father", {}).get("value"),
        "motherId": r.get("mother", {}).get("value"),
    })

df_parents = pd.DataFrame(rows)

filename = "wikidata_ca_people_null_birthyear_parents.csv"
df_parents.to_csv(filename, index=False)

print(f"Saved {filename}, rows: {len(df_parents)} ✅")


Fetching CA citizens with NULL birthyear...
Saved wikidata_ca_people_null_birthyear_parents.csv, rows: 179 ✅


## Relationship - Relatives

### US

In [ ]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# 2️⃣ Create 10-year ranges from 1900 to 2026
year_ranges = [(year, min(year + 9, 2026)) for year in range(1900, 2026, 5)]

for start_year, end_year in year_ranges:
    print(f"Fetching data for birth years {start_year} to {end_year}...")

    # 3️⃣ SPARQL query for this decade
    query = f"""
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX p:    <http://www.wikidata.org/prop/>
    PREFIX ps:   <http://www.wikidata.org/prop/statement/>
    PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      (STRAFTER(STR(?person),   "entity/") AS ?personId)
      (STRAFTER(STR(?relative), "entity/") AS ?relativeId)
      (STRAFTER(STR(?kinship),  "entity/") AS ?kinshipId)
      ?kinshipLabel
    WHERE {{
      ?person wdt:P31 wd:Q5 ;
              wdt:P27 wd:Q30 ; -- US citizen
              wdt:P345 ?imdb ;
              wdt:P569 ?birthDate .

      # Use the statement form so we can read qualifiers
      ?person p:P1038 ?relStmt .
      ?relStmt ps:P1038 ?relative .

      # kinship qualifier (cousin/aunt/etc.) — may be missing
      OPTIONAL {{ ?relStmt pq:P1039 ?kinship . }}

      hint:Prior hint:rangeSafe true.

      FILTER("{start_year}-01-01T00:00:00Z"^^xsd:dateTime <= ?birthDate)
      FILTER(?birthDate <= "{end_year}-12-31T23:59:59Z"^^xsd:dateTime)

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en" .
      }}
    }}
    ORDER BY ?personId ?relativeId ?kinshipId
    LIMIT 200000
    """

    sparql.setQuery(query)

    # 4️⃣ Execute
    results = sparql.query().convert()

    # 5️⃣ Parse results
    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "personId": r["personId"]["value"],
            "relativeId": r["relativeId"]["value"],
            "kinshipId": r.get("kinshipId", {}).get("value", ""),           # blank if no qualifier
            "kinshipLabel": r.get("kinshipLabel", {}).get("value", ""),    # blank if no label
        })

    # 6️⃣ Convert to DataFrame
    df_relatives = pd.DataFrame(rows)

    # 7️⃣ Save CSV for this decade
    filename = f"wikidata_us_people_relatives_{start_year}_{end_year}.csv"
    df_relatives.to_csv(filename, index=False)

    print(f"Saved {filename}, rows: {len(df_relatives)}\n")


Fetching data for birth years 1900 to 1909...
Saved wikidata_us_people_relatives_1900_1909.csv, rows: 117

Fetching data for birth years 1905 to 1914...
Saved wikidata_us_people_relatives_1905_1914.csv, rows: 103

Fetching data for birth years 1910 to 1919...
Saved wikidata_us_people_relatives_1910_1919.csv, rows: 111

Fetching data for birth years 1915 to 1924...
Saved wikidata_us_people_relatives_1915_1924.csv, rows: 139

Fetching data for birth years 1920 to 1929...
Saved wikidata_us_people_relatives_1920_1929.csv, rows: 158

Fetching data for birth years 1925 to 1934...
Saved wikidata_us_people_relatives_1925_1934.csv, rows: 178

Fetching data for birth years 1930 to 1939...
Saved wikidata_us_people_relatives_1930_1939.csv, rows: 165

Fetching data for birth years 1935 to 1944...
Saved wikidata_us_people_relatives_1935_1944.csv, rows: 199

Fetching data for birth years 1940 to 1949...
Saved wikidata_us_people_relatives_1940_1949.csv, rows: 270

Fetching data for birth years 1945 to

### CA

In [ ]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

# 2️⃣ Create 10-year ranges from 1900 to 2026
year_ranges = [(year, min(year + 9, 2026)) for year in range(1900, 2026, 5)]

for start_year, end_year in year_ranges:
    print(f"Fetching data for birth years {start_year} to {end_year}...")

    # 3️⃣ SPARQL query for this decade
    query = f"""
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX wd:   <http://www.wikidata.org/entity/>
    PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
    PREFIX p:    <http://www.wikidata.org/prop/>
    PREFIX ps:   <http://www.wikidata.org/prop/statement/>
    PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
    PREFIX hint: <http://www.bigdata.com/queryHints#>

    SELECT DISTINCT
      (STRAFTER(STR(?person),   "entity/") AS ?personId)
      (STRAFTER(STR(?relative), "entity/") AS ?relativeId)
      (STRAFTER(STR(?kinship),  "entity/") AS ?kinshipId)
      ?kinshipLabel
    WHERE {{
      ?person wdt:P31 wd:Q5 ;
              wdt:P27 wd:Q16 ; -- CA citizen
              wdt:P345 ?imdb ;
              wdt:P569 ?birthDate .

      # Use the statement form so we can read qualifiers
      ?person p:P1038 ?relStmt .
      ?relStmt ps:P1038 ?relative .

      # kinship qualifier (cousin/aunt/etc.) — may be missing
      OPTIONAL {{ ?relStmt pq:P1039 ?kinship . }}

      hint:Prior hint:rangeSafe true.

      FILTER("{start_year}-01-01T00:00:00Z"^^xsd:dateTime <= ?birthDate)
      FILTER(?birthDate <= "{end_year}-12-31T23:59:59Z"^^xsd:dateTime)

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en" .
      }}
    }}
    ORDER BY ?personId ?relativeId ?kinshipId
    LIMIT 200000
    """

    sparql.setQuery(query)

    # 4️⃣ Execute
    results = sparql.query().convert()

    # 5️⃣ Parse results
    rows = []
    for r in results["results"]["bindings"]:
        rows.append({
            "personId": r["personId"]["value"],
            "relativeId": r["relativeId"]["value"],
            "kinshipId": r.get("kinshipId", {}).get("value", ""),           # blank if no qualifier
            "kinshipLabel": r.get("kinshipLabel", {}).get("value", ""),    # blank if no label
        })

    # 6️⃣ Convert to DataFrame
    df_relatives = pd.DataFrame(rows)

    # 7️⃣ Save CSV for this decade
    filename = f"wikidata_ca_people_relatives_{start_year}_{end_year}.csv"
    df_relatives.to_csv(filename, index=False)

    print(f"Saved {filename}, rows: {len(df_relatives)}\n")


### birthyear null - CA

In [89]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

print("Fetching CA citizens with IMDb + relatives, where birthyear is NULL...")

query = """
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX p:    <http://www.wikidata.org/prop/>
PREFIX ps:   <http://www.wikidata.org/prop/statement/>
PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
PREFIX hint: <http://www.bigdata.com/queryHints#>

SELECT DISTINCT
  (STRAFTER(STR(?person),   "entity/") AS ?personId)
  (STRAFTER(STR(?relative), "entity/") AS ?relativeId)
  (STRAFTER(STR(?kinship),  "entity/") AS ?kinshipId)
  ?kinshipLabel
WHERE {
  ?person wdt:P31 wd:Q5 ;
          wdt:P27 wd:Q16 ;      # Canada citizen only
          wdt:P345 ?imdb .      # IMDb required

  # 🚨 Birthdate does NOT exist
  FILTER NOT EXISTS { ?person wdt:P569 ?birthDate }

  # Relative statements
  ?person p:P1038 ?relStmt .
  ?relStmt ps:P1038 ?relative .

  # Optional kinship qualifier
  OPTIONAL { ?relStmt pq:P1039 ?kinship . }

  hint:Prior hint:rangeSafe true.

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "en" .
  }
}
ORDER BY ?personId ?relativeId ?kinshipId
LIMIT 200000
"""

sparql.setQuery(query)

# Execute
results = sparql.query().convert()

# Parse results
rows = []
for r in results["results"]["bindings"]:
    rows.append({
        "personId": r["personId"]["value"],
        "relativeId": r["relativeId"]["value"],
        "kinshipId": r.get("kinshipId", {}).get("value", ""),
        "kinshipLabel": r.get("kinshipLabel", {}).get("value", ""),
    })

df_relatives = pd.DataFrame(rows)

filename = "wikidata_ca_people_relatives_null_birthyear.csv"
df_relatives.to_csv(filename, index=False)

print(f"Saved {filename}, rows: {len(df_relatives)} ✅")


Fetching CA citizens with IMDb + relatives, where birthyear is NULL...
Saved wikidata_ca_people_relatives_null_birthyear.csv, rows: 4 ✅


### birthyear null - US

In [88]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# 1️⃣ SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

print("Fetching US citizens with IMDb + relatives, where birthyear is NULL...")

query = """
PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
PREFIX wd:   <http://www.wikidata.org/entity/>
PREFIX p:    <http://www.wikidata.org/prop/>
PREFIX ps:   <http://www.wikidata.org/prop/statement/>
PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
PREFIX hint: <http://www.bigdata.com/queryHints#>

SELECT DISTINCT
  (STRAFTER(STR(?person),   "entity/") AS ?personId)
  (STRAFTER(STR(?relative), "entity/") AS ?relativeId)
  (STRAFTER(STR(?kinship),  "entity/") AS ?kinshipId)
  ?kinshipLabel
WHERE {
  ?person wdt:P31 wd:Q5 ;
          wdt:P27 wd:Q30 ;      # US citizen only
          wdt:P345 ?imdb .      # IMDb required

  # 🚨 Birthdate does NOT exist
  FILTER NOT EXISTS { ?person wdt:P569 ?birthDate }

  # Relative statements
  ?person p:P1038 ?relStmt .
  ?relStmt ps:P1038 ?relative .

  # Optional kinship qualifier
  OPTIONAL { ?relStmt pq:P1039 ?kinship . }

  hint:Prior hint:rangeSafe true.

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "en" .
  }
}
ORDER BY ?personId ?relativeId ?kinshipId
LIMIT 200000
"""

sparql.setQuery(query)

# Execute
results = sparql.query().convert()

# Parse results
rows = []
for r in results["results"]["bindings"]:
    rows.append({
        "personId": r["personId"]["value"],
        "relativeId": r["relativeId"]["value"],
        "kinshipId": r.get("kinshipId", {}).get("value", ""),
        "kinshipLabel": r.get("kinshipLabel", {}).get("value", ""),
    })

df_relatives = pd.DataFrame(rows)

filename = "wikidata_us_people_relatives_null_birthyear.csv"
df_relatives.to_csv(filename, index=False)

print(f"Saved {filename}, rows: {len(df_relatives)} ✅")


Fetching US citizens with IMDb + relatives, where birthyear is NULL...
Saved wikidata_us_people_relatives_null_birthyear.csv, rows: 51 ✅


## Movie award (+ IMDB id)

In [ ]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON, POST, POSTDIRECTLY
import pandas as pd
import time

# ----------------------------
# Config
# ----------------------------
MOVIE_PATH = "processed/title_movie.csv"
TV_PATH    = "processed/title_tvSeries.csv"
OUT_PATH   = "processed/title_awards_noms.csv"

BATCH_SIZE = 100  # IMDb lookup heavier → smaller batch safer

# ----------------------------
# SPARQL client
# ----------------------------
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.setMethod(POST)
sparql.setRequestMethod(POSTDIRECTLY)

def qid(uri: str) -> str:
    return uri.rsplit("/", 1)[-1]

def run_query_safe(q: str, sleep_s: float = 0.6, retries: int = 5):
    sparql.setQuery(q)

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            res = sparql.query().convert()
            time.sleep(sleep_s)
            return res
        except Exception as e:
            last_err = e
            wait = min(30, 2 ** attempt)
            print(f"Retry {attempt}/{retries} after {wait}s...")
            time.sleep(wait)

    raise last_err

# ----------------------------
# Step 1: Load IMDb IDs
# ----------------------------
movie_df = pd.read_csv(MOVIE_PATH, usecols=["tconst"])
tv_df = pd.read_csv(TV_PATH, usecols=["tconst"])

imdb_ids = pd.concat([
    movie_df["tconst"],
    tv_df["tconst"]
]).dropna().astype(str).drop_duplicates().tolist()

print(f"Total IMDb IDs loaded: {len(imdb_ids):,}")

# ----------------------------
# Step 2: Query Wikidata
# ----------------------------
rows = []

for i in range(0, len(imdb_ids), BATCH_SIZE):
    batch = imdb_ids[i:i + BATCH_SIZE]
    values_block = " ".join(f'"{imdb}"' for imdb in batch)

    print(f"Processing batch {i//BATCH_SIZE + 1}...")

    query = f"""
    PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
    PREFIX p:    <http://www.wikidata.org/prop/>
    PREFIX ps:   <http://www.wikidata.org/prop/statement/>
    PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
    PREFIX bd:   <http://www.bigdata.com/rdf#>
    PREFIX wikibase: <http://wikiba.se/ontology#>

    SELECT ?film ?filmLabel ?imdb ?type ?award ?awardLabel ?awardDate WHERE {{
      VALUES ?imdb {{ {values_block} }}

      ?film wdt:P345 ?imdb .

      {{
        ?film p:P166 ?stmt .
        ?stmt ps:P166 ?award .
        OPTIONAL {{ ?stmt pq:P585 ?awardDate . }}
        BIND("award" AS ?type)
      }}
      UNION
      {{
        ?film p:P1411 ?stmt .
        ?stmt ps:P1411 ?award .
        OPTIONAL {{ ?stmt pq:P585 ?awardDate . }}
        BIND("nomination" AS ?type)
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    res = run_query_safe(query)
    bindings = res["results"]["bindings"]

    for b in bindings:
        film_uri = b["film"]["value"]
        award_uri = b["award"]["value"]
        award_date = b.get("awardDate", {}).get("value")
        award_year = award_date[:4] if award_date else None

        rows.append({
            "film_wikiId": qid(film_uri),
            "film_name": b.get("filmLabel", {}).get("value"),
            "imdb_id": b.get("imdb", {}).get("value"),
            "type": b.get("type", {}).get("value"),
            "award_wikiId": qid(award_uri),
            "award_name": b.get("awardLabel", {}).get("value"),
            "award_year": award_year
        })

# ----------------------------
# Step 3: Deduplicate
# ----------------------------
df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(
        subset=["film_wikiId", "type", "award_wikiId", "award_year"],
        keep="first"
    ).reset_index(drop=True)

# ----------------------------
# Step 4: Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)

print(f"\nSaved {OUT_PATH}")
print(f"Total rows: {len(df):,}")


Total IMDb IDs loaded: 212,316
Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Processing batch 5...
Processing batch 6...
Processing batch 7...
Processing batch 8...
Processing batch 9...
Processing batch 10...
Processing batch 11...
Processing batch 12...
Processing batch 13...
Processing batch 14...
Processing batch 15...
Processing batch 16...
Processing batch 17...
Processing batch 18...
Processing batch 19...
Processing batch 20...
Processing batch 21...
Processing batch 22...
Processing batch 23...
Processing batch 24...
Processing batch 25...
Processing batch 26...
Processing batch 27...
Processing batch 28...
Processing batch 29...
Processing batch 30...
Processing batch 31...
Processing batch 32...
Processing batch 33...
Processing batch 34...
Processing batch 35...
Processing batch 36...
Processing batch 37...
Processing batch 38...
Processing batch 39...
Processing batch 40...
Processing batch 41...
Processing batch 42...
Processing b

## People Award


In [95]:
# !pip install SPARQLWrapper pandas

from SPARQLWrapper import SPARQLWrapper, JSON, POST, POSTDIRECTLY
import pandas as pd
import time
from pathlib import Path

# ----------------------------
# Config
# ----------------------------
PERSON_CSV_PATH = Path("processed/person.csv")     # change if needed
# OUT_INFO_PATH   = Path("processed/person_info_from_personcsv.csv")
OUT_AWARD_PATH  = Path("processed/person_awards_noms.csv")

BATCH_SIZE_INFO  = 300   # labels/bio data can be bigger batches
BATCH_SIZE_AWARD = 120   # awards/nom usually heavier; keep smaller

# ----------------------------
# SPARQL client
# ----------------------------
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.setMethod(POST)
sparql.setRequestMethod(POSTDIRECTLY)

def qid(uri: str) -> str:
    return uri.rsplit("/", 1)[-1]

def run_query_safe(q: str, sleep_s: float = 0.8, retries: int = 5):
    """
    WDQS sometimes returns HTML (429/503) or truncates huge responses.
    This wrapper retries and prints response snippet if it isn't JSON.
    """
    sparql.setQuery(q)
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            res = sparql.query().convert()
            time.sleep(sleep_s)
            return res
        except Exception as e:
            last_err = e
            # Try to extract raw response (often HTML) for debugging
            try:
                raw = sparql.query().response.read().decode("utf-8", errors="replace")
                print("\n⚠️ WDQS returned non-JSON / error page. Snippet:\n")
                print(raw[:800])
                print("\n...(truncated)...\n")
            except Exception:
                pass

            wait = min(30, 2 ** attempt)  # exponential backoff up to 30s
            print(f"Retry {attempt}/{retries} after {wait}s...\n")
            time.sleep(wait)

    raise last_err

# ----------------------------
# Load people from person.csv
# ----------------------------
df_people = pd.read_csv(PERSON_CSV_PATH)

if "person" not in df_people.columns:
    raise ValueError(f"'person' column not found in {PERSON_CSV_PATH}. Columns: {list(df_people.columns)}")

person_uris = (
    df_people["person"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.str.contains("wikidata.org/entity/", na=False)]
    .drop_duplicates()
    .tolist()
)

print(f"Loaded {len(person_uris)} unique Wikidata person URIs from {PERSON_CSV_PATH}")

# ----------------------------
# Fetch person info for those URIs
# ----------------------------
def fetch_person_info(person_uris, batch_size: int = 300):
    """
    Returns dataframe with:
      person_wikiId, person_uri, name, gender, birthDate, citizenship, imdb
    """
    rows = []

    for i in range(0, len(person_uris), batch_size):
        batch = person_uris[i:i + batch_size]
        values_block = " ".join(f"<{u}>" for u in batch)

        info_query = f"""
        PREFIX wd:   <http://www.wikidata.org/entity/>
        PREFIX wdt:  <http://www.wikidata.org/prop/direct/>
        PREFIX bd:   <http://www.bigdata.com/rdf#>
        PREFIX wikibase: <http://wikiba.se/ontology#>

        SELECT ?person ?personLabel ?genderLabel ?birthDate ?citizenshipLabel ?imdb WHERE {{
          VALUES ?person {{ {values_block} }}

          OPTIONAL {{ ?person wdt:P21  ?gender. }}
          OPTIONAL {{ ?person wdt:P569 ?birthDate. }}
          OPTIONAL {{ ?person wdt:P27  ?citizenship. }}
          OPTIONAL {{ ?person wdt:P345 ?imdb. }}

          SERVICE wikibase:label {{
            bd:serviceParam wikibase:language "en".
          }}
        }}
        """

        res = run_query_safe(info_query)
        bindings = res["results"]["bindings"]

        for b in bindings:
            person_uri = b["person"]["value"]
            rows.append({
                "person_wikiId": qid(person_uri),
                "person_uri": person_uri,
                "name": b.get("personLabel", {}).get("value"),
                "gender": b.get("genderLabel", {}).get("value"),
                "birthDate": b.get("birthDate", {}).get("value"),
                "citizenship": b.get("citizenshipLabel", {}).get("value"),
                "imdb": b.get("imdb", {}).get("value"),
            })

        print(f"  info batch {i//batch_size + 1}: +{len(bindings)} rows")

    df = pd.DataFrame(rows)

    # de-dupe (some people can appear multiple times if multiple citizenship values, etc.)
    if not df.empty:
        df = df.drop_duplicates(
            subset=["person_wikiId", "person_uri", "name", "gender", "birthDate", "citizenship", "imdb"],
            keep="first"
        ).reset_index(drop=True)

    return df

# ----------------------------
# Fetch awards + nominations for those URIs
# ----------------------------
def fetch_awards_for_people(person_uris, batch_size: int = 120):
    """
    Return dataframe with columns:
      person_wikiId, type, award_wikiId, award_name, award_year
    """
    rows = []

    for i in range(0, len(person_uris), batch_size):
        batch = person_uris[i:i + batch_size]
        values_block = " ".join(f"<{u}>" for u in batch)

        awards_query = f"""
        PREFIX p:    <http://www.wikidata.org/prop/>
        PREFIX ps:   <http://www.wikidata.org/prop/statement/>
        PREFIX pq:   <http://www.wikidata.org/prop/qualifier/>
        PREFIX bd:   <http://www.bigdata.com/rdf#>
        PREFIX wikibase: <http://wikiba.se/ontology#>

        SELECT ?person ?type ?award ?awardLabel ?awardDate WHERE {{
          VALUES ?person {{ {values_block} }}

          {{
            ?person p:P166 ?stmt .
            ?stmt ps:P166 ?award .
            OPTIONAL {{ ?stmt pq:P585 ?awardDate . }}
            BIND("award" AS ?type)
          }}
          UNION
          {{
            ?person p:P1411 ?stmt .
            ?stmt ps:P1411 ?award .
            OPTIONAL {{ ?stmt pq:P585 ?awardDate . }}
            BIND("nomination" AS ?type)
          }}

          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        """

        res = run_query_safe(awards_query)
        bindings = res["results"]["bindings"]

        for b in bindings:
            person_uri = b["person"]["value"]
            award_uri = b["award"]["value"]
            award_date = b.get("awardDate", {}).get("value")
            award_year = award_date[:4] if award_date and len(award_date) >= 4 else None

            rows.append({
                "person_wikiId": qid(person_uri),
                "type": b["type"]["value"],  # award / nomination
                "award_wikiId": qid(award_uri),
                "award_name": b.get("awardLabel", {}).get("value"),
                "award_year": award_year
            })

        print(f"  award batch {i//batch_size + 1}: +{len(bindings)} rows")

    df = pd.DataFrame(rows)

    # dedupe
    if not df.empty:
        df = df.drop_duplicates(
            subset=["person_wikiId", "type", "award_wikiId", "award_year"],
            keep="first"
        ).reset_index(drop=True)

    return df

# ----------------------------
# Run
# ----------------------------
# df_info = fetch_person_info(person_uris, batch_size=BATCH_SIZE_INFO)
# df_info.to_csv(OUT_INFO_PATH, index=False)
# print(f"Saved person info -> {OUT_INFO_PATH} rows={len(df_info)}")

df_awards = fetch_awards_for_people(person_uris, batch_size=BATCH_SIZE_AWARD)
df_awards.to_csv(OUT_AWARD_PATH, index=False)
print(f"Saved awards/noms -> {OUT_AWARD_PATH} rows={len(df_awards)}")


Loaded 112718 unique Wikidata person URIs from processed/person.csv
  award batch 1: +88 rows
  award batch 2: +47 rows
  award batch 3: +73 rows
  award batch 4: +97 rows
  award batch 5: +105 rows
  award batch 6: +85 rows
  award batch 7: +79 rows
  award batch 8: +87 rows
  award batch 9: +157 rows
  award batch 10: +65 rows
  award batch 11: +68 rows
  award batch 12: +105 rows
  award batch 13: +170 rows
  award batch 14: +39 rows
  award batch 15: +57 rows
  award batch 16: +87 rows
  award batch 17: +46 rows
  award batch 18: +61 rows
  award batch 19: +48 rows
  award batch 20: +58 rows
  award batch 21: +51 rows
  award batch 22: +102 rows
  award batch 23: +26 rows
  award batch 24: +75 rows
  award batch 25: +284 rows
  award batch 26: +170 rows
  award batch 27: +94 rows
  award batch 28: +195 rows
  award batch 29: +43 rows
  award batch 30: +35 rows
  award batch 31: +156 rows
  award batch 32: +54 rows
  award batch 33: +120 rows
  award batch 34: +57 rows
  award batch

## Merge


In [81]:
import pandas as pd
import glob

def merge_csvs(pattern, output_filename, required_columns=None):
    dfs = []
    files = glob.glob(pattern)

    if not files:
        print("No files matched pattern.")
        return

    for f in files:
        try:
            df = pd.read_csv(f)

            # Skip empty files (no columns)
            if df.shape[1] == 0:
                print(f"Skipping {f} (no columns)")
                continue

            # Optional: skip if required columns missing
            if required_columns:
                missing = [col for col in required_columns if col not in df.columns]
                if missing:
                    print(f"Skipping {f} (missing columns: {missing})")
                    continue

            dfs.append(df)

        except Exception as e:
            print(f"Skipping {f} (error: {e})")
            continue

    if not dfs:
        print("No valid files to merge.")
        return

    # Merge
    merged_df = pd.concat(dfs, ignore_index=True)

    # Remove duplicates
    merged_df = merged_df.drop_duplicates(keep="first")

    merged_df.to_csv(output_filename, index=False)

    print(f"Saved {output_filename} ({len(merged_df)} rows) ✅")


In [99]:
pattern = r"person_1.csv"
output_filename = "processed/person.csv"

merge_csvs(pattern, output_filename)


Saved processed/person.csv (116245 rows) ✅
